# XTraffic ST-GNN — Fusion training (free T4 GPU)

Phase 6: retrains METR-LA with the **three real heterogeneous modalities**
(weather + events + transit) and lays the fusion model side by side with the
traffic-only baseline. Also **regenerates `metr_la_best.pt`** (the traffic-only
checkpoint) — handy since a local smoke run clobbered the old placeholder.

**Runtime → Change runtime type → T4 GPU** before running. Edit `REPO_URL` below.


In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


In [ ]:
REPO_URL = 'https://github.com/<your-username>/OlympiFlow.git'  # <-- EDIT ME
!git clone $REPO_URL
%cd OlympiFlow


In [ ]:
# Pinned lightweight deps (training needs torch + numpy + pandas + pyyaml + matplotlib + requests).
!pip install -q pyyaml==6.0.1 requests==2.31.0 pandas matplotlib scipy


## 1. Build the traffic tensors + the three modality sidecars
Everything is downloaded/derived reproducibly — no manual files. The sidecars
are windowed and split *identically* to the traffic tensors, so sample i lines
up across all feeds.


In [ ]:
# Traffic tensors (Phase 1). Produces processed/metr_la/{train,val,test}.npz
!python -m xtraffic.data.pipelines.metr_la


In [ ]:
# Weather (Open-Meteo ERA5, keyless). NOTE: the archive API does not serve
# `visibility` -> that channel is all-NaN and gets zeroed (warned); temp+precip
# carry the signal. This is handled defensively, not a crash.
!python -m xtraffic.data.pipelines.weather --dataset metr_la


In [ ]:
# Events (committed curated 2012 venue schedule, proximity-decayed).
!python -m xtraffic.data.pipelines.events --dataset metr_la


In [ ]:
# Transit (LA Metro GTFS static -> nearby stop count per node).
!python -m xtraffic.data.pipelines.transit --dataset metr_la


## 2. Train the traffic-only baseline
Uses `train_metr_la.yaml` (`use_sidecars` off) -> `metr_la_best.pt`. This is the
apples-to-apples baseline AND restores the traffic-only checkpoint the rest of
the pipeline (Phases 3–5) loads.


In [ ]:
!python -m xtraffic.models.gnn.train --config configs/train_metr_la.yaml


## 3. Train the fusion model
Uses `train_metr_la_fusion.yaml` (`use_sidecars: true`, `run_name: metr_la_fusion`)
-> `metr_la_fusion_best.pt`. Same architecture and hyperparameters as the baseline;
the ONLY difference is the three modality feeds, so any gap is attributable to fusion.


In [ ]:
!python -m xtraffic.models.gnn.train --config configs/train_metr_la_fusion.yaml


## 4. Fusion vs traffic-only comparison table
Evaluates both checkpoints on the same test split at 15/30/60 min and reports the
learned per-modality gates (how much the model trusts each feed). Saved to
`evaluation/results/fusion/comparison.{csv,json}`.


In [ ]:
!python -m xtraffic.evaluation.fusion_comparison --dataset metr_la


## 5. (Optional) Baselines for the full prediction table
Historical-average + linear-regression reference numbers on the same splits.


In [ ]:
!python -m xtraffic.models.gnn.baselines --config configs/train_metr_la.yaml


## 6. Download the trained checkpoints + comparison table


In [ ]:
from google.colab import files
for p in ['xtraffic/models/gnn/checkpoints/metr_la_best.pt',
          'xtraffic/models/gnn/checkpoints/metr_la_fusion_best.pt',
          'xtraffic/evaluation/results/fusion/comparison.csv',
          'xtraffic/evaluation/results/fusion/comparison.json']:
    files.download(p)
